In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/dataset.json
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/1259.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/659.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/274.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/315.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/919.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/2037.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/1819.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/1938.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/948.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/123.tif
/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions/imgs/imgs/959.tif
/kaggle/input/datasets/soumyashreedas9/uc

In [2]:
import os
import sys
 
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TRANSFORMERS_NO_ADVISORY_WARNINGS'] = '1'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
 
import warnings
import logging
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)
logging.getLogger('tensorflow').setLevel(logging.ERROR)
 
import io
stderr = sys.stderr
sys.stderr = io.StringIO()
 
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import (
    VisionEncoderDecoderModel,
    ViTImageProcessor,
    GPT2TokenizerFast,
    GPT2Config,
    get_cosine_schedule_with_warmup
)
from PIL import Image
import numpy as np
from tqdm import tqdm
import random
import matplotlib.pyplot as plt
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction
import gc
from datetime import datetime
import re
 
sys.stderr = stderr
 
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
 
gc.collect()
torch.cuda.empty_cache()
 
start_time = datetime.now()

In [3]:
BASE_PATH = '/kaggle/input/datasets/soumyashreedas9/ucm-model/UCM_captions'
IMG_DIR   = os.path.join(BASE_PATH, 'imgs/imgs')          # folder containing all UCM .tif/.jpg images
JSON_FILE = os.path.join(BASE_PATH, 'dataset.json')  # UCM captions file
 
# Training hyperparameters (matched to your paper's Table I — UCM column)
BATCH_SIZE            = 24             # effective batch = 48 with grad accumulation
EPOCHS                = 25
LR                    = 5e-5
WARMUP_RATIO          = 0.15
GRADIENT_ACCUMULATION = 2
LABEL_SMOOTHING       = 0.1
WEIGHT_DECAY          = 0.01
 
# Generation parameters (UCM column in Table I)
MAX_LENGTH         = 30
MIN_LENGTH         = 6
NUM_BEAMS          = 4
LENGTH_PENALTY     = 1.0
NO_REPEAT_NGRAM    = 2
REPETITION_PENALTY = 1.2
TEMPERATURE        = 1.0
 
# Data configuration
USE_AUGMENTATION   = True
AUGMENTATION_PROB  = 0.3
USE_ALL_CAPTIONS   = True   # use all 5 reference captions per image for training
CAPTIONS_PER_IMAGE = 5
 
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
 
print(f"Device: {DEVICE}")
print(f"Batch Size: {BATCH_SIZE} x {GRADIENT_ACCUMULATION} = {BATCH_SIZE * GRADIENT_ACCUMULATION} effective")
print(f"Epochs: {EPOCHS}")
print(f"Learning Rate: {LR} (cosine schedule)")
print(f"Label Smoothing: {LABEL_SMOOTHING}")
print(f"Max Length: {MAX_LENGTH}")
print(f"Num Beams: {NUM_BEAMS}")
print(f"Data Augmentation: {USE_AUGMENTATION} (prob={AUGMENTATION_PROB})")
print(f"Captions Per Image: ALL 5")
print(f"Repetition Penalty: {REPETITION_PENALTY}")

Device: cuda
Batch Size: 24 x 2 = 48 effective
Epochs: 50
Learning Rate: 5e-05 (cosine schedule)
Label Smoothing: 0.1
Max Length: 30
Num Beams: 4
Data Augmentation: True (prob=0.3)
Captions Per Image: ALL 5
Repetition Penalty: 1.2


In [4]:
 
with open(JSON_FILE) as f:
    data = json.load(f)
 
image_captions = {}
for img in data['images']:
    img_path = os.path.join(IMG_DIR, img['filename'])
    if os.path.exists(img_path):
        captions = [sent['raw'].lower().strip() for sent in img['sentences']]
        image_captions[img_path] = captions
 
print(f"\nDataset Statistics:")
print(f"   Total Images: {len(image_captions)}")
 
all_images = list(image_captions.keys())
random.shuffle(all_images)
 
# 80/10/10 split — standard for UCM-Captions, matches your paper's Section IV-A
n_train = int(0.80 * len(all_images))
n_val   = int(0.10 * len(all_images))
train_images = all_images[:n_train]
val_images   = all_images[n_train:n_train + n_val]
test_images  = all_images[n_train + n_val:]
 
print(f"   Train: {len(train_images)} | Val: {len(val_images)} | Test: {len(test_images)}")


Dataset Statistics:
   Total Images: 2100
   Train: 1680 | Val: 210 | Test: 210


In [5]:
tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
tokenizer.add_special_tokens({'pad_token': '<pad>'})
feature_extractor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
 
print("Tokenizer loaded")
print("Feature extractor loaded")
 

Tokenizer loaded
Feature extractor loaded


In [6]:
class UCMDataset(Dataset):
    def __init__(self, image_paths, image_captions, augment=False):
        self.image_paths = image_paths
        self.image_captions = image_captions
        self.augment = augment
 
        if USE_ALL_CAPTIONS:
            self.samples = []
            for img_path in image_paths:
                for caption in image_captions[img_path]:
                    self.samples.append((img_path, caption))
        else:
            self.samples = []
            for img_path in image_paths:
                captions = image_captions[img_path]
                selected = random.sample(captions, min(CAPTIONS_PER_IMAGE, len(captions)))
                for caption in selected:
                    self.samples.append((img_path, caption))
 
    def __len__(self):
        return len(self.samples)
 
    def augment_image(self, img):
        if not self.augment or random.random() > AUGMENTATION_PROB:
            return img
        if random.random() > 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        if random.random() > 0.6:
            from PIL import ImageEnhance
            if random.random() > 0.5:
                enhancer = ImageEnhance.Brightness(img)
                factor = random.uniform(0.8, 1.2)
                img = enhancer.enhance(factor)
            else:
                enhancer = ImageEnhance.Contrast(img)
                factor = random.uniform(0.8, 1.2)
                img = enhancer.enhance(factor)
        return img
 
    def __getitem__(self, idx):
        img_path, caption = self.samples[idx]
 
        img = Image.open(img_path).convert('RGB')
        img = self.augment_image(img)
 
        pixel_values = feature_extractor(img, return_tensors='pt').pixel_values[0]
 
        tokens = tokenizer(
            caption,
            max_length=MAX_LENGTH,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
 
        labels = tokens['input_ids'][0].clone()
        labels[labels == tokenizer.pad_token_id] = -100
 
        return pixel_values, labels, img_path
 
 
def collate_fn(batch):
    pixels = torch.stack([b[0] for b in batch])
    labels = torch.stack([b[1] for b in batch])
    paths = [b[2] for b in batch]
    return pixels, labels, paths
 
 
train_dataset = UCMDataset(train_images, image_captions, augment=USE_AUGMENTATION)
val_dataset   = UCMDataset(val_images, image_captions, augment=False)
 
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=4, collate_fn=collate_fn, pin_memory=True,
    drop_last=True, prefetch_factor=2
)
 
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=4, collate_fn=collate_fn, pin_memory=True,
    prefetch_factor=2
)
 
print(f"Data Loaders:")
print(f"   Train Batches: {len(train_loader)}")
print(f"   Val Batches: {len(val_loader)}")
print(f"   Training Samples: {len(train_dataset)} (all 5 captions per image)")

Data Loaders:
   Train Batches: 350
   Val Batches: 44
   Training Samples: 8400 (all 5 captions per image)


In [7]:

class EnhancedCaptioningModel(nn.Module):
    def __init__(self):
        super().__init__()
 
        decoder_config = GPT2Config.from_pretrained('gpt2')
        decoder_config.is_decoder = True
        decoder_config.add_cross_attention = True
 
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            self.model = VisionEncoderDecoderModel.from_encoder_decoder_pretrained(
                'google/vit-base-patch16-224-in21k',
                'gpt2',
                decoder_config=decoder_config,
            )
 
        self.model.decoder.resize_token_embeddings(len(tokenizer), mean_resizing=False)
 
        self.model.config.decoder_start_token_id = tokenizer.bos_token_id
        self.model.config.pad_token_id           = tokenizer.pad_token_id
        self.model.config.eos_token_id           = tokenizer.eos_token_id
        self.model.config.use_cache              = False
 
        # NOTE: newer `transformers` versions forbid setting generation
        # parameters (max_length, num_beams, etc.) on model.config directly —
        # they must live on model.generation_config instead. Setting them on
        # model.config raises: "You have modified the pretrained model
        # configuration to control generation... This strategy is not
        # supported anymore."
        self.model.generation_config.decoder_start_token_id = tokenizer.bos_token_id
        self.model.generation_config.pad_token_id           = tokenizer.pad_token_id
        self.model.generation_config.eos_token_id           = tokenizer.eos_token_id
        self.model.generation_config.max_length             = MAX_LENGTH
 
        self.dropout = nn.Dropout(0.1)
 
    def forward(self, pixel_values, labels=None):
        outputs = self.model(pixel_values=pixel_values, labels=labels)
 
        if labels is not None and self.training and LABEL_SMOOTHING > 0:
            logits     = outputs.logits
            vocab_size = logits.size(-1)
 
            confidence      = 1.0 - LABEL_SMOOTHING
            smoothing_value = LABEL_SMOOTHING / (vocab_size - 1)
 
            one_hot = torch.zeros_like(logits)
            one_hot.scatter_(-1, labels.unsqueeze(-1).clamp(min=0), 1.0)
            one_hot = one_hot * confidence + (1 - one_hot) * smoothing_value
 
            mask    = (labels != -100).unsqueeze(-1).float()
            one_hot = one_hot * mask
 
            log_probs    = F.log_softmax(logits, dim=-1)
            loss         = -(one_hot * log_probs).sum(dim=-1)
            loss         = loss.sum() / mask.sum()
            outputs.loss = loss
 
        return outputs
 
    def generate(self, pixel_values, **kwargs):
        kwargs.update({
            'use_cache'           : False,
            'pad_token_id'        : tokenizer.pad_token_id,
            'eos_token_id'        : tokenizer.eos_token_id,
            'max_length'          : MAX_LENGTH,
            'min_length'          : MIN_LENGTH,
            'no_repeat_ngram_size': NO_REPEAT_NGRAM,
            'repetition_penalty'  : REPETITION_PENALTY,
            'length_penalty'      : LENGTH_PENALTY,
            'early_stopping'      : True,
            'num_beams'           : NUM_BEAMS,
        })
        return self.model.generate(pixel_values, **kwargs)
 
 
model  = EnhancedCaptioningModel().to(DEVICE)
scaler = torch.cuda.amp.GradScaler()
 
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
 
print(f"\nModel Architecture:")
print(f"   Total Parameters    : {total_params:,}")
print(f"   Trainable Parameters: {trainable_params:,}")
print(f"   Mixed Precision     : Enabled")

Loading weights:   0%|          | 0/200 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Model Architecture:
   Total Parameters    : 239,196,672
   Trainable Parameters: 239,196,672
   Mixed Precision     : Enabled


In [8]:
optimizer = AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
    betas=(0.9, 0.999),
    eps=1e-8
)
 
total_steps  = (len(train_loader) // GRADIENT_ACCUMULATION) * EPOCHS
warmup_steps = int(WARMUP_RATIO * total_steps)
 
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)
 
print(f"Optimizer Configuration:")
print(f"   Optimizer: AdamW")
print(f"   Learning Rate: {LR}")
print(f"   Total Steps: {total_steps}")
print(f"   Warmup Steps: {warmup_steps}")

Optimizer Configuration:
   Optimizer: AdamW
   Learning Rate: 5e-05
   Total Steps: 8750
   Warmup Steps: 1312


In [9]:
def train_epoch(model, loader, optimizer, scheduler, epoch):
    model.train()
    total_loss = 0
    optimizer.zero_grad()
 
    pbar = tqdm(loader, desc=f"Epoch {epoch:02d}/{EPOCHS}")
 
    for batch_idx, (pixel_values, labels, _) in enumerate(pbar):
        pixel_values = pixel_values.to(DEVICE)
        labels = labels.to(DEVICE)
 
        with torch.cuda.amp.autocast():
            outputs = model(pixel_values=pixel_values, labels=labels)
            loss = outputs.loss / GRADIENT_ACCUMULATION
 
        scaler.scale(loss).backward()
 
        if (batch_idx + 1) % GRADIENT_ACCUMULATION == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
 
        total_loss += loss.item() * GRADIENT_ACCUMULATION
        current_lr = optimizer.param_groups[0]['lr']
        pbar.set_postfix({
            'loss': f'{loss.item() * GRADIENT_ACCUMULATION:.3f}',
            'lr': f'{current_lr:.2e}'
        })
 
    if len(loader) % GRADIENT_ACCUMULATION != 0:
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad()
 
    return total_loss / len(loader)
 
 
def clean_caption(text):
    text = ' '.join(text.split())
    text = re.sub(r'\.{2,}', '.', text)
    text = re.sub(r',{2,}', ',', text)
    text = re.sub(r'\s+([.,!?])', r'\1', text)
    text = re.sub(r'([.,!?])([^\s])', r'\1 \2', text)
    text = text.rstrip('. ')
    if '. .' in text:
        text = text.split('. .')[0]
    if text and not text.endswith('.'):
        text += '.'
    return text.strip()
 
 
def validate(model, val_loader, image_captions):
    model.eval()
    predictions = []
    references = []
 
    with torch.no_grad():
        pbar = tqdm(val_loader, desc="  Validating", leave=False)
        for pixel_values, _, img_paths in pbar:
            pixel_values = pixel_values.to(DEVICE)
            generated = model.generate(pixel_values)
 
            for i, gen_ids in enumerate(generated):
                pred = tokenizer.decode(gen_ids, skip_special_tokens=True)
                pred = clean_caption(pred)
                pred_tokens = pred.lower().split()
 
                refs = image_captions[img_paths[i]]
                ref_tokens = [ref.lower().split() for ref in refs]
 
                predictions.append(pred_tokens)
                references.append(ref_tokens)
 
    smooth = SmoothingFunction()
    bleu1 = corpus_bleu(references, predictions, weights=(1,0,0,0), smoothing_function=smooth.method1)
    bleu2 = corpus_bleu(references, predictions, weights=(0.5,0.5,0,0), smoothing_function=smooth.method1)
    bleu3 = corpus_bleu(references, predictions, weights=(0.33,0.33,0.33,0), smoothing_function=smooth.method1)
    bleu4 = corpus_bleu(references, predictions, weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth.method1)
 
    return bleu1, bleu2, bleu3, bleu4

In [10]:
history = {
    'loss': [], 'bleu1': [], 'bleu2': [], 'bleu3': [], 'bleu4': [], 'epochs': []
}
 
best_bleu = 0.0
best_epoch = 0
patience = 10   # matches Table I: UCM early-stop patience = 10 epochs
patience_counter = 0
 
for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(model, train_loader, optimizer, scheduler, epoch)
    history['loss'].append(loss)
 
    if epoch % 2 == 0 or epoch == 1:
        bleu1, bleu2, bleu3, bleu4 = validate(model, val_loader, image_captions)
        history['bleu1'].append(bleu1)
        history['bleu2'].append(bleu2)
        history['bleu3'].append(bleu3)
        history['bleu4'].append(bleu4)
        history['epochs'].append(epoch)
 
        status = []
        if bleu4 > best_bleu:
            improvement = bleu4 - best_bleu
            best_bleu = bleu4
            best_epoch = epoch
            torch.save(model.state_dict(), 'best_model_ucm.pth')
            status.append(f"SAVED (+{improvement*100:.2f}%)")
            patience_counter = 0
        else:
            patience_counter += 1
 
        if bleu4 >= 0.80:
            status.append("EXCELLENT!")
        elif bleu4 >= 0.75:
            status.append("GREAT!")
        elif bleu4 >= 0.70:
            status.append("GOOD!")
 
        status_str = " ".join(status) if status else ""
        print(f"Epoch {epoch:02d} | Loss: {loss:.4f} | "
              f"BLEU: {bleu1:.3f} {bleu2:.3f} {bleu3:.3f} {bleu4:.3f} ({bleu4*100:.2f}%) {status_str}")
 
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered at epoch {epoch}")
            print(f"   No improvement for {patience} validation cycles")
            break
    else:
        print(f"Epoch {epoch:02d} | Loss: {loss:.4f}")
 
    if epoch % 5 == 0:
        gc.collect()
        torch.cuda.empty_cache()
 
 

Epoch 01/50: 100%|██████████| 350/350 [02:10<00:00,  2.69it/s, loss=4.207, lr=6.67e-06] 


Epoch 01 | Loss: 30.9844 | BLEU: 0.332 0.198 0.110 0.038 (3.75%) SAVED (+3.75%)


Epoch 02/50: 100%|██████████| 350/350 [02:08<00:00,  2.72it/s, loss=2.890, lr=1.33e-05]


Epoch 02 | Loss: 3.2829 | BLEU: 0.425 0.341 0.293 0.254 (25.35%) SAVED (+21.60%)


Epoch 03/50: 100%|██████████| 350/350 [02:08<00:00,  2.72it/s, loss=2.266, lr=2.00e-05]


...

Epoch 42 | Loss: 1.5395 | BLEU: 0.814 0.783 0.755 0.725 (72.47%) GOOD!


Epoch 43/50: 100%|██████████| 350/350 [02:09<00:00,  2.71it/s, loss=1.541, lr=3.27e-06]


Epoch 43 | Loss: 1.5397


Epoch 44/50: 100%|██████████| 350/350 [02:09<00:00,  2.71it/s, loss=1.548, lr=2.42e-06]


Epoch 44 | Loss: 1.5382 | BLEU: 0.816 0.783 0.753 0.721 (72.07%) GOOD!

Early stopping triggered at epoch 44
   No improvement for 10 validation cycles


In [11]:
import subprocess
 
import nltk
for pkg in ['wordnet', 'omw-1.4', 'punkt', 'punkt_tab']:
    try:
        nltk.download(pkg, quiet=True)
        
    except Exception:
        pass
 
try:
    from rouge_score import rouge_scorer as rouge_lib
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'rouge-score', '-q'])
    from rouge_score import rouge_scorer as rouge_lib
 
try:
    from pycocoevalcap.cider.cider import Cider as CiderScorer
except ImportError:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install',
                           'pycocoevalcap', 'pycocotools', '-q'])
    from pycocoevalcap.cider.cider import Cider as CiderScorer
 
from nltk.translate.meteor_score import meteor_score as nltk_meteor
 
model.load_state_dict(torch.load('best_model_ucm.pth'))
model.eval()
 
test_predictions  = []
test_references   = []
test_pred_strings = []
test_ref_strings  = []
 
print("Generating predictions for test set...")
with torch.no_grad():
    for img_path in tqdm(test_images, desc="Testing"):
        img          = Image.open(img_path).convert('RGB')
        pixel_values = feature_extractor(img, return_tensors='pt').pixel_values.to(DEVICE)
 
        generated = model.generate(pixel_values)
        pred      = tokenizer.decode(generated[0], skip_special_tokens=True)
        pred      = clean_caption(pred)
 
        refs = image_captions[img_path]
 
        test_predictions.append(pred.lower().split())
        test_references.append([ref.lower().split() for ref in refs])
        test_pred_strings.append(pred.lower())
        test_ref_strings.append([ref.lower() for ref in refs])
 
# ─── BLEU ───────────────────────────────────────────────
smooth     = SmoothingFunction()
test_bleu1 = corpus_bleu(test_references, test_predictions, weights=(1,0,0,0), smoothing_function=smooth.method1)
test_bleu2 = corpus_bleu(test_references, test_predictions, weights=(0.5,0.5,0,0), smoothing_function=smooth.method1)
test_bleu3 = corpus_bleu(test_references, test_predictions, weights=(0.33,0.33,0.33,0), smoothing_function=smooth.method1)
test_bleu4 = corpus_bleu(test_references, test_predictions, weights=(0.25,0.25,0.25,0.25), smoothing_function=smooth.method1)
 
# ─── METEOR ─────────────────────────────────────────────
meteor_scores = []
for pred_toks, ref_tok_list in zip(test_predictions, test_references):
    score = nltk_meteor(ref_tok_list, pred_toks)
    meteor_scores.append(score)
test_meteor = float(np.mean(meteor_scores))
 
# ─── ROUGE-L ────────────────────────────────────────────
scorer = rouge_lib.RougeScorer(['rougeL'], use_stemmer=True)
rougeL_scores = []
for pred_str, ref_str_list in zip(test_pred_strings, test_ref_strings):
    best_f1 = max(scorer.score(ref, pred_str)['rougeL'].fmeasure for ref in ref_str_list)
    rougeL_scores.append(best_f1)
test_rougeL = float(np.mean(rougeL_scores))
 
# ─── CIDEr ──────────────────────────────────────────────
cider_gts  = {i: ref_list   for i, ref_list in enumerate(test_ref_strings)}
cider_res  = {i: [pred_str] for i, pred_str in enumerate(test_pred_strings)}
cider_eval = CiderScorer()
test_cider, _ = cider_eval.compute_score(cider_gts, cider_res)
 
# ─── PRINT SUMMARY ──────────────────────────────────────
print(f"\n{'='*60}")
print(f"  UCM-CAPTIONS TEST SET PERFORMANCE")
print(f"{'='*60}")
print(f"  BLEU-1  : {test_bleu1:.4f}  ({test_bleu1*100:.2f}%)")
print(f"  BLEU-2  : {test_bleu2:.4f}  ({test_bleu2*100:.2f}%)")
print(f"  BLEU-3  : {test_bleu3:.4f}  ({test_bleu3*100:.2f}%)")
print(f"  BLEU-4  : {test_bleu4:.4f}  ({test_bleu4*100:.2f}%)")
print(f"  METEOR  : {test_meteor:.4f}  ({test_meteor*100:.2f}%)")
print(f"  ROUGE-L : {test_rougeL:.4f}  ({test_rougeL*100:.2f}%)")
print(f"  CIDEr   : {test_cider:.4f}")
print(f"{'='*60}")
 
elapsed = datetime.now() - start_time
print(f"\nTotal runtime: {elapsed}")
print(f"Best epoch: {best_epoch} | Best val BLEU-4: {best_bleu:.4f}")

Generating predictions for test set...


Testing: 100%|██████████| 210/210 [01:55<00:00,  1.81it/s]



  UCM-CAPTIONS TEST SET PERFORMANCE
  BLEU-1  : 0.8100  (81.00%)
  BLEU-2  : 0.7733  (77.33%)
  BLEU-3  : 0.7428  (74.28%)
  BLEU-4  : 0.7090  (70.90%)
  METEOR  : 0.7892  (78.92%)
  ROUGE-L : 0.8728  (87.28%)
  CIDEr   : 2.9739

Total runtime: 4:07:11.693108
Best epoch: 24 | Best val BLEU-4: 0.7366
